In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [0]:
data=pd.read_csv("Bank Customer Churn Prediction.csv")
data.head(10)


In [0]:
data.products_number.max()

In [0]:
data.info()

Removing irrelvent columns or features

In [0]:
data = data.drop(columns=['country'])

In [0]:
X=data.iloc[:,:-1]
Y=data.iloc[:,-1]

In [0]:
X.nunique()

In [0]:
categorical_cols=[x for x in X.columns if X[x].dtype=='object']
numeric_cols=[x for x in X.columns if X[x].dtype!='object']

numeric_cols

In [0]:
X[numeric_cols].describe()

In [0]:
var=['credit_card','active_member','products_number']
for i in var:
    numeric_cols.remove(i)
    categorical_cols.append(i)


In [0]:
categorical_cols,numeric_cols

In [0]:
plt.figure(figsize=(10,8))
plt.xticks(rotation=45)
sns.heatmap(X[numeric_cols].corr(),annot=True)

In [0]:
labels = 'Churn', 'Not Churn'
sizes = [data.churn[data['churn']==1].count(), data.churn[data['churn']==0].count()]
explode = (0, 0.1)
fig1, ax1 = plt.subplots(figsize=(8, 6))
ax1.pie(sizes, explode=explode,colors=('red','green'),labels=labels, autopct='%1.1f%%',shadow=True, startangle=90)
ax1.axis('equal')
plt.title("churned vs Not churn", size = 20)
plt.show()

In [0]:
X[numeric_cols].hist(figsize=(15,12))
plt.legend()

In [0]:
fig,axarr=plt.subplots(2,2,figsize=(12,10))
cnt=0
for i in range(2):
    for j in range(2):
        sns.boxplot(x=numeric_cols[cnt],hue='churn',data=data,ax=axarr[i][j])
        cnt=cnt+1

creditscore  boxplot shows some outliers, let's see this

In [0]:
lower=data.credit_score.quantile(q=0.005)
upper=data.credit_score.quantile(q=0.995)
data['credit_score']=np.where(data['credit_score']>upper,upper,np.where(data['credit_score']<lower,lower,data['credit_score']))

sns.boxplot(data['credit_score'])

In [0]:
list=['gender','credit_card','active_member']
fig,axarr=plt.subplots(3,1,figsize=(12,10))
cnt=0
for i, category in enumerate(list):
    sns.countplot(x=category, hue='churn', data=data, ax=axarr[i])
    axarr[i].set_xlabel(category)
    axarr[i].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [0]:
fig, axarr = plt.subplots(3, 2, figsize=(20, 12))
sns.boxplot(y='credit_score',x = 'churn', hue = 'churn',data = data, ax=axarr[0][0])
sns.boxplot(y='age',x = 'churn', hue = 'churn',data = data , ax=axarr[0][1])
sns.boxplot(y='tenure',x = 'churn', hue = 'churn',data = data, ax=axarr[1][0])
sns.boxplot(y='balance',x = 'churn', hue = 'churn',data = data, ax=axarr[1][1])
sns.boxplot(y='products_number',x = 'churn', hue = 'churn',data = data, ax=axarr[2][0])
sns.boxplot(y='estimated_salary',x = 'churn', hue = 'churn',data = data, ax=axarr[2][1])

In [0]:
data.groupby('products_number')['estimated_salary'].mean()

In [0]:
print(pd.pivot_table(data,index='churn',columns='products_number',values='customer_id',aggfunc='count'))


# Feature engineering

 Categorize customers into age groups (e.g., young, middle-aged, senior). This can help capture differences in behavior and risk across different age ranges.

In [0]:
data['age_class'] = np.where(data['age'] <=35, 'young',
                           np.where((data['age'] > 35) & (data['age'] < 55), 'middle_aged',
                                     np.where((data['age'] >= 55), 'senior', data['age']
                                              )
                                        )
                                    )
                           
                      


In [0]:
data['age_class'].value_counts()

In [0]:
print(pd.pivot_table(data,index='churn',columns='age_class',values='customer_id',aggfunc='count'))

In [0]:
# data['seniority'] = np.where(data['tenure'] <=2,'new',
#                            np.where((data['tenure'] >2) & (data['tenure'] <=6 ), 'intermediate',
#                                      np.where((data['tenure'] >6 ), 'long-standing', data['tenure']
#                                               )
#                                         ))
                            

In [0]:
# print(pd.pivot_table(data,index='churn',columns='seniority',values='customer_id',aggfunc='count'))

In [0]:
data_train = data.sample(frac=0.85,random_state=41)
data_test = data.drop(data_train.index)
print(len(data_train))

print(len(data_test))

In [0]:
data_test

In [0]:
# data_train['balance_salaryRatio'] = data_train.balance/data_train.estimated_salary
# sns.boxplot(y='balance_salaryRatio',x = 'churn', hue = 'churn',data = data_train)
# plt.ylim(-1, 5)

In [0]:
print(pd.pivot_table(data,index='churn',columns='age_class',values='customer_id',aggfunc='count'))

we saw that the salary has little effect on the probability of a customer churning. However as seen above, the ratio of the bank balance and the estimated salary indicates that customers with a higher balance salary ratio churn more.

In [0]:
data_train['TenureByAge'] = data_train.tenure/(data_train.age)
sns.boxplot(y='TenureByAge',x = 'churn', hue = 'churn',data = data_train)
plt.ylim(-1, 1)
plt.show()



In [0]:
sns.distplot(data_train['TenureByAge'])

Given that tenure is a 'function' of age, we introduce a variable aiming to standardize tenure over age:

In [0]:
data_train['CreditScoreGivenAge'] = data_train.credit_score/(data_train.age)

In [0]:
sns.boxplot(y='CreditScoreGivenAge',x='churn',hue='churn',data=data_train)


Lastly we introduce a variable to capture credit score given age to take into account credit behavior

In [0]:
data_train.head().columns

In [0]:
# numeric_cols=['credit_score','age','tenure','balance','products_number','estimated_salary','balance_salaryRatio','TenureByAge','CreditScoreGivenAge']
# categorical_cols=['credit_card','active_member','country','gender','age_class','seniority']
numeric_cols=['credit_score','age','tenure','balance','products_number','estimated_salary','TenureByAge','CreditScoreGivenAge']
# categorical_cols=['credit_card','active_member','gender','age_class','seniority']
categorical_cols=['credit_card','active_member','gender','age_class']
data_train = data_train[['churn'] + numeric_cols + categorical_cols]
data_train.head()

For the one hot variables, we change 0 to -1 so that the models can capture a negative relation 
where the attribute in inapplicable instead of 0

In [0]:
data_train.loc[data_train.credit_card == 0, 'credit_card'] = -1
data_train.loc[data_train.active_member == 0, 'active_member'] = -1
data_train.head()

In [0]:
remove =[]
for i in categorical_cols:
    if (data_train[i].dtype == 'str' or data_train[i].dtype == 'object'):
        for j in data_train[i].unique():
            data_train[i+'_'+j] = np.where(data_train[i] == j,1,-1)
        remove.append(i)
data_train = data_train.drop(remove, axis=1)
data_train.head()


In [0]:
# minVec = data_train[numeric_cols].min().copy()
# maxVec = data_train[numeric_cols].max().copy()
# data_train[numeric_cols] = (data_train[numeric_cols]-minVec)/(maxVec-minVec)
# data_train.columns

from sklearn.preprocessing import MinMaxScaler
# Initialize the Min-Max scaler
scaler = MinMaxScaler()

# Fit and transform the data using the scaler
scaler.fit(data_train[numeric_cols])
data_train[numeric_cols]= scaler.transform(data_train[numeric_cols])
# scaled_data now contains the scaled values



In [0]:
data_train.columns


In [0]:
# data prep pipeline for test data
def dataTestPipeline(df_predict,df_train_Cols):

#     df_predict['balance_salaryRatio'] = df_predict.balance/df_predict.estimated_salary
    df_predict['TenureByAge'] = df_predict.tenure/(df_predict.age)
    df_predict['CreditScoreGivenAge'] = df_predict.credit_score/(df_predict.age)

#     continuous_vars = ['credit_score','age','tenure','balance','products_number','estimated_salary','balance_salaryRatio',
#                    'TenureByAge','CreditScoreGivenAge']
#     cat_vars = ['credit_card','active_member',"country", "gender","age_class","seniority"]
    continuous_vars = ['credit_score','age','tenure','balance','products_number','estimated_salary',
                   'TenureByAge','CreditScoreGivenAge']
#     cat_vars = ['credit_card','active_member', "gender","age_class","seniority"]
    cat_vars = ['credit_card','active_member', "gender","age_class"]

    df_predict = df_predict[['churn'] + continuous_vars + cat_vars]
    # Change the 0 in categorical variables to -1
    df_predict.loc[df_predict.credit_card == 0, 'HasCrCard'] = -1
    df_predict.loc[df_predict.active_member == 0, 'IsActiveMember'] = -1
    # One hot encode the categorical variables
#     lst = [ "gender","age_class","seniority"]
    lst = [ "gender","age_class"]
    remove = []
    for i in lst:
        for j in df_predict[i].unique():
            df_predict[i+'_'+j] = np.where(df_predict[i] == j,1,-1)
        remove.append(i)
    df_predict = df_predict.drop(remove, axis=1)
#     # Ensure that all one hot encoded variables that appear in the train data appear in the subsequent data
    L = [set(df_train_Cols) - set(df_predict.columns)]
    print(L)
    for l in L:
        df_predict[str(l)] = -1        

#     df_predict[continuous_vars] = (df_predict[continuous_vars]-minVec)/(maxVec-minVec)
    df_predict[continuous_vars] =scaler.transform(df_predict[continuous_vars])

    df_predict = df_predict[df_train_Cols]
    return df_predict

data_test = dataTestPipeline(data_test,data_train.columns)

In [0]:
# import joblib

# joblib.dump(scaler, 'your_scaler.pkl')
scaler.data_min_

In [0]:
# data_test = dataTestPipeline(data_test,data_train.columns,minVec,maxVec)
# data_test = data_test.mask(np.isinf(data_test))
# data_test = data_test.dropna()
data_test.min()


In [0]:
data_train.head()

In [0]:
x_train=data_train.iloc[:,1:]
y_train=data_train.iloc[:,0]

In [0]:
x_train

In [0]:
x_test=data_test.iloc[:,1:]
y_test=data_test.iloc[:,0]


In [0]:
# Support functions
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score,cross_val_predict
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from scipy.stats import uniform

# Fit models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB


In [0]:
from sklearn.metrics import f1_score,recall_score,precision_score,confusion_matrix


Naive Bayes Model

In [0]:
classifier = GaussianNB()
param_grid = {
    'var_smoothing': [1e-7, 1e-6, 1e-5]
}
grid_search = GridSearchCV(estimator=classifier, param_grid=param_grid, cv=3)

# Perform the hyperparameter tuning on the training data
grid_search.fit(x_train, y_train)


best_params = grid_search.best_params_
best_model = grid_search.best_estimator_



In [0]:
best_params

In [0]:
# custom_priors = [0.21, 0.79] 
classifierGNB = GaussianNB(var_smoothing=1e-07)
classifierGNB.fit(x_train, y_train)

In [0]:
scores=cross_val_score(classifierGNB,x_train,y_train,cv=5,scoring='f1')
scores

In [0]:
resGNB=classifierGNB.predict(x_test)
correct_predictions = sum(p == t for p, t in zip(resGNB, y_test))

# Calculate the total number of predictions
total_predictions = y_test.shape[0]

# Calculate the accuracy
accuracy = correct_predictions / total_predictions

print("Correct Predictions:", correct_predictions)
print("Total Predictions:", total_predictions)
print("Accuracy:", accuracy)




In [0]:
sns.heatmap(confusion_matrix(y_test,resGNB),annot=True,fmt='d')

In [0]:
def best_model(model):
    print(model.best_score_)    
    print(model.best_params_)
    print(model.best_estimator_)
def get_auc_scores(y_actual, method,method2):
    auc_score = roc_auc_score(y_actual, method); 
    fpr_df, tpr_df, _ = roc_curve(y_actual, method2); 
    return (auc_score, fpr_df, tpr_df)

In [0]:
# Fit primal logistic regression
param_grid = {'C': [0.1,0.5,1,10,50,100], 'max_iter': [800,1000,1200],'intercept_scaling':[1],'penalty':['l2','l1'], 'tol':[0.00001,0.0001,0.000001],'solver':['sag','newton-cg','lbfgs','liblinear']}
log_primal_Grid = GridSearchCV(LogisticRegression(),param_grid, cv=5, verbose=1,n_jobs=-1,scoring='f1')
log_primal_Grid.fit(x_train,y_train)
best_model(log_primal_Grid)

In [0]:
ModelLogistic=LogisticRegression(C=100, max_iter=800, penalty='l1', solver='liblinear')
ModelLogistic.fit(x_train,y_train)

In [0]:
# logpr=log_primal_Grid.predict(x_test)
scores=ModelLogistic.predict(x_train)
# scores
print(f1_score(scores,y_train),recall_score(scores,y_train),precision_score(scores,y_train))


In [0]:
sns.heatmap(confusion_matrix(scores,y_train),annot=True,fmt='d')

In [0]:
Logisticres=ModelLogistic.predict(x_test)
sns.heatmap(confusion_matrix(y_test,Logisticres),annot=True,fmt='d')

In [0]:
correct_predictions = sum(p == t for p, t in zip(scores, y_test))

# Calculate the total number of predictions
total_predictions = y_test.shape[0]

# Calculate the accuracy
accuracy = correct_predictions / total_predictions

print("Correct Predictions:", correct_predictions)
print("Total Predictions:", total_predictions)
print("Accuracy:", accuracy)


In [0]:
# Fit logistic regression with degree 2 polynomial kernel
param_grid = {'C': [0.1,0.5,1,10], 'max_iter': [800,1000,1200],'intercept_scaling':[1],'penalty':['l1'],
              'tol':[0.0001,0.000001]}
poly2 = PolynomialFeatures(degree=2)
df_train_pol2 = poly2.fit_transform(x_train)
# Create the logistic regression model
logistic_regression = LogisticRegression(solver='liblinear')

# Create a GridSearchCV object
log_pol2_Grid = GridSearchCV(logistic_regression, param_grid, cv=3, refit=True, verbose=1,n_jobs=-1,scoring='f1')

# Fit the model on the polynomial features
log_pol2_Grid.fit(df_train_pol2, y_train)
best_model(log_pol2_Grid)


In [0]:
mod2LGR=LogisticRegression(C=10, max_iter=800, penalty='l1', solver='liblinear',tol=1e-06)
mod2LGR.fit(df_train_pol2, y_train)

In [0]:
scores=mod2LGR.predict(df_train_pol2)
# scores
print(f1_score(scores,y_train),recall_score(scores,y_train),precision_score(scores,y_train))


In [0]:
# Fit logistic regression with degree 2 polynomial kernel
param_grid = {'C': [0.1,0.5,1,10], 'max_iter': [800,1000,1200],'intercept_scaling':[1],'penalty':['l1'],
              'tol':[0.0001,0.000001]}
poly3 = PolynomialFeatures(degree=3)
df_train_pol3 = poly3.fit_transform(x_train)
# Create the logistic regression model
logistic_regression = LogisticRegression(solver='liblinear')

# Create a GridSearchCV object
log_pol3_Grid = GridSearchCV(logistic_regression, param_grid, cv=3, refit=True, verbose=1,n_jobs=-1,scoring='f1')

# Fit the model on the polynomial features
log_pol3_Grid.fit(df_train_pol3, y_train)
best_model(log_pol3_Grid)


In [0]:

# Create the logistic regression model

mod3LGR=LogisticRegression(C=10, max_iter=800, penalty='l1', solver='liblinear')
mod3LGR.fit(df_train_pol3, y_train)

In [0]:
x_train

In [0]:
scores1=mod3LGR.predict_proba(df_train_pol3)
# scores
# print(f1_score(scores1,y_train),recall_score(scores1,y_train),precision_score(scores1,y_train))



In [0]:
import joblib
joblib.dump(svmrbf, 'logisticregressiondegree3.pkl')

In [0]:
scores1

In [0]:
binary_predictions = (scores1[:, 1] >= 0.30).astype(int)

In [0]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Assuming you have your logistic regression model (log_pol2_Grid) trained and the test data (x_test, y_test)
# Get predicted probabilities for the positive class (class 1)
predicted_probabilities = mod3LGR.predict_proba(df_train_pol3)[:, 1]

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_train, predicted_probabilities)

# Calculate AUC (Area Under the Curve)
auc = roc_auc_score(y_train, predicted_probabilities)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()


In [0]:
sns.heatmap(confusion_matrix(binary_predictions,y_train),annot=True,fmt='d')
print(f1_score(binary_predictions,y_train),recall_score(binary_predictions,y_train),precision_score(binary_predictions,y_train))



In [0]:
# Transform test data with the same polynomial features
df_test_pol2 = poly2.transform(x_test)

# Make predictions on the test data
predictions = log_pol2_Grid.predict(df_test_pol2)

In [0]:
sns.heatmap(confusion_matrix(predictions,y_test),annot=True,fmt='d')

In [0]:
# Transform test data with the same polynomial features
df_test_pol3 = poly3.transform(x_test)

# Make predictions on the test data
predictions = log_pol3_Grid.predict_proba(df_test_pol3)

In [0]:
binary_predictionsTest = (predictions[:, 1] >= 0.3).astype(int)

In [0]:
sns.heatmap(confusion_matrix(binary_predictionsTest,y_test),annot=True,fmt='d')

In [0]:
print(f1_score(binary_predictionsTest,y_test),recall_score(binary_predictionsTest,y_test),precision_score(binary_predictionsTest,y_test))


In [0]:
scores=log_pol2_Grid.predict(x_train)
# scores
print(f1_score(scores,y_train),recall_score(scores,y_train),precision_score(scores,y_train))
sns.heatmap(confusion_matrix(scores,y_train),annot=True,fmt='d')

In [0]:
# Fit SVM with pol kernel
param_grid = {'C': [1,10,50,100], 'gamma': [0.1,0.01,0.001],'probability':[True],'kernel': ['poly'],'degree':[3,4] }
SVM_grid = GridSearchCV(SVC(), param_grid, cv=5, refit=True, verbose=1,n_jobs=-1,scoring='f1')
SVM_grid.fit(x_train,y_train)


In [0]:
best_model(SVM_grid)

In [0]:
SVM3poly=SVC(C=100, degree=4, gamma=0.1, kernel='poly', probability=True)
SVM3poly.fit(x_train,y_train)

In [0]:
scores=SVM3poly.predict(x_train)
# scores
print(f1_score(scores,y_train),recall_score(scores,y_train),precision_score(scores,y_train))
sns.heatmap(confusion_matrix(scores,y_train),annot=True,fmt='d')

In [0]:
predsvmpoly=SVM3poly.predict(x_test)
# correct_predictions = sum(p == t for p, t in zip(svmres, y_test))

# # Calculate the total number of predictions
# total_predictions = y_test.shape[0]

# # Calculate the accuracy
# accuracy = correct_predictions / total_predictions

# print("Correct Predictions:", correct_predictions)
# print("Total Predictions:", total_predictions)
# print("Accuracy:", accuracy)
sns.heatmap(confusion_matrix(predsvmpoly,y_test),annot=True,fmt='d')

In [0]:
# Fit SVM with RBF Kernel
param_grid = {'C': [0.5,1,10,100], 'gamma': [0.1,0.01,0.001],'probability':[True],'kernel': ['rbf']}
svm_grid = GridSearchCV(SVC(), param_grid, cv=3, refit=True, verbose=1,n_jobs=-1,scoring='f1')
svm_grid.fit(x_train,y_train)
best_model(svm_grid)

In [0]:
svmrbf=SVC(C=100, gamma=0.1, probability=True)
svmrbf.fit(x_train,y_train)

In [0]:
import joblib
joblib.dump(svmrbf, 'SVM_rbf.pkl')

In [0]:
svmprob=(svmrbf.predict_proba(x_train)[:, 1] >= 0.30).astype(int)

In [0]:

# scores
print(f1_score(svmprob,y_train),recall_score(svmprob,y_train),precision_score(svmprob,y_train))
sns.heatmap(confusion_matrix(svmprob,y_train),annot=True,fmt='d')

In [0]:
predsvmrbf=svmrbf.predict(x_test)

sns.heatmap(confusion_matrix(predsvmrbf,y_test),annot=True,fmt='d')

In [0]:
param_grid = {'max_depth': [5, 6, 7, 8,10],'n_estimators':[100,120,150],'min_samples_split': [5, 6, 7],'max_features':['auto','log2'],'max_leaf_nodes':[8,10,12],'max_samples':[0.8]}
RanFor_grid = GridSearchCV(RandomForestClassifier(n_jobs=-1,oob_score=True), param_grid, cv=5, verbose=1,scoring='f1')
RanFor_grid.fit(x_train,y_train)
best_model(RanFor_grid)

In [0]:
RanFor_Grid=RandomForestClassifier(max_depth=10, max_features='log2', max_leaf_nodes=12,
                       max_samples=0.8, min_samples_split=5, n_estimators=150,
                       n_jobs=-1, oob_score=True)
RanFor_Grid.fit(x_train,y_train)
oob_score = RanFor_Grid.oob_score_
print("OOB Score:", oob_score)

In [0]:
scores=RanFor_Grid.predict(x_train)
# scores
print(f1_score(scores,y_train),recall_score(scores,y_train),precision_score(scores,y_train))
sns.heatmap(confusion_matrix(scores,y_train),annot=True,fmt='d')

In [0]:
param_grid = {'max_depth': [ 6, 7, 8,10,12],'n_estimators':[100,150],'min_samples_split': [5, 6, 7],'max_features':['auto','log2'],'max_leaf_nodes':[8,10,12,14],'max_samples':[0.8]}
RanFor_rancv=RandomizedSearchCV(RandomForestClassifier(n_jobs=-1,oob_score=True),param_grid,n_iter=100,cv=5,n_jobs=-1,verbose=1)
RanFor_rancv.fit(x_train,y_train)
best_model(RanFor_rancv)

In [0]:
RanFor_Rancv=RandomForestClassifier(max_depth=12, max_leaf_nodes=14, max_samples=0.8,
                       min_samples_split=5, n_estimators=150, n_jobs=-1,
                       oob_score=True)
RanFor_Rancv.fit(x_train,y_train)
oob_score = RanFor_Rancv.oob_score_
print("OOB Score:", oob_score)

In [0]:
scores=RanFor_Rancv.predict(x_train)
# scores
print(f1_score(scores,y_train),recall_score(scores,y_train),precision_score(scores,y_train))
sns.heatmap(confusion_matrix(scores,y_train),annot=True,fmt='d')

In [0]:
scores=RanFor_grid.predict(x_train)
# scores
print(f1_score(scores,y_train),recall_score(scores,y_train),precision_score(scores,y_train))
sns.heatmap(confusion_matrix(scores,y_train),annot=True,fmt='d')

In [0]:
pred=RanFor_grid.predict(x_test)
correct_predictions = sum(p == t for p, t in zip(pred, y_test))

# Calculate the total number of predictions
total_predictions = y_test.shape[0]

# Calculate the accuracy
accuracy = correct_predictions / total_predictions

print("Correct Predictions:", correct_predictions)
print("Total Predictions:", total_predictions)
print("Accuracy:", accuracy)


In [0]:
pip install xgboost

In [0]:
import xgboost as xgb

In [0]:
model=xgb.XGBClassifier()

In [0]:
model.fit(x_train,y_train) 

In [0]:
# param_grid = {
#     'learning_rate': [0.1, 0.01, 0.001],
#     'max_depth': [3, 5, 7, 8],
#     'n_estimators': [150, 200, 250, 300, 350, 400],
#     'subsample': [0.6, 0.7, 0.8],
#     'min_child_weight': [2, 5, 10, 15, 20],
#     'gamma': [0.1, 0.2, 0.01, 0.03],
#     'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
#     'colsample_bylevel': [0.6, 0.7, 0.8, 1.0],
#     'reg_lambda': [0.01, 0.1, 1, 5, 10, 20, 50],
#     'reg_alpha': [0.1, 1, 5, 10, 20, 50],
#     'booster':['gbtree', 'gblinear','dart'],
#     'sample_type':['uniform','weighted'],
#     'normalize_type':['tree','forest'],
#     'objective':['binary:logistic'],
#     'eval_metric':['f1']

# }

param_grid = {
    'learning_rate': [0.1, 0.01, 0.001],
    'max_depth': [3, 5, 7, 8],
    'n_estimators': [150, 200, 250, 300, 350, 400],
    'subsample': [0.6, 0.7, 0.8],
    'min_child_weight': [2, 5, 10, 15, 20],
    'gamma': [0.1, 0.2, 0.01, 0.03],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'colsample_bylevel': [0.6, 0.7, 0.8, 1.0],
    'reg_lambda': [0.01, 0.1, 1, 5, 10, 20, 50],
    'reg_alpha': [0.1, 1, 5, 10, 20, 50],
    'objective': ['binary:logistic'],
    'eval_metric': ['f1']
}



XG=GridSearchCV(model,param_grid,n_jobs=-1,cv=5,verbose=1,error_score='raise')
XG.fit(x_train,y_train)
best_model(XG)

In [0]:
import xgboost as xgb


params = {
    'subsample': 0.7,
    'reg_lambda': 1,
    'reg_alpha': 10,
    'n_estimators': 300,
    'min_child_weight': 20,
    'max_depth': 3,
    'learning_rate': 0.1,
    'gamma': 0.1,
    'colsample_bytree': 1.0,
    'colsample_bylevel': 0.8,
    'booster': 'dart'
}
XGB = xgb.XGBClassifier(**params)
XGB.fit(x_train, y_train)


In [0]:
# scores=cross_val_predict(,x_train,y_train,cv=5,method='predict')
score=XGB.predict(x_train)
print(f1_score(scores,y_train),recall_score(scores,y_train),precision_score(scores,y_train))
sns.heatmap(confusion_matrix(scores,y_train),annot=True,fmt='d')

In [0]:
sns.heatmap(confusion_matrix(scores,y_train),annot=True,fmt='d')

In [0]:
from sklearn.model_selection import train_test_split
# Step 1: Create a holdout set
X_train, X_holdout, Y_train, Y_holdout = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

# Step 2: Select and train base models
XGB1=xgb.XGBClassifier(learning_rate=0.1,max_depth=3,n_estimators=120)
RanFor_1=RandomForestClassifier(n_estimators=120,max_depth=10, max_features='log2', max_leaf_nodes=14,
                       max_samples=0.8, min_samples_split=6, n_jobs=-1,
                       oob_score=True)

XGB1.fit(X_train,Y_train)
RanFor_1.fit(X_train,Y_train)
poly3 = PolynomialFeatures(degree=3)
df_train_pol3 = poly3.fit_transform(X_train)

mod3LGR=LogisticRegression(C=0.5, max_iter=1000, penalty='l1', solver='liblinear')
mod3LGR.fit(df_train_pol3, Y_train)


# Step 4: Generate predictions from base models
predictions_base1 = XGB1.predict(X_holdout)
predictions_base2 = RanFor_1.predict(X_holdout)
df_test_pol3 = poly3.fit_transform(X_holdout)
base3=mod3LGR.predict(df_test_pol3)
# Step 5: Concatenate predictions
meta_features = np.column_stack((predictions_base1, predictions_base2,base3))




In [0]:
# Step 6: Train the meta-model
param_grid = {'C': [0.5,1,10,100], 'gamma': [0.1,0.01,0.001],'probability':[True],'kernel': ['rbf']}
svm_grid = GridSearchCV(SVC(), param_grid, cv=3, refit=True, verbose=1)
svm_grid.fit(meta_features,Y_holdout)
best_model(svm_grid)

In [0]:
meta_model =SVC(C=0.5, gamma=0.1, probability=True)
meta_model.fit(meta_features, Y_holdout)
df_test_pol3 = poly3.fit_transform(x_test)
base3=mod3LGR.predict(df_test_pol3)
# Step 7: Evaluate the stacked ensemble
meta_features_test = np.column_stack((XGB1.predict(x_test), RanFor_1.predict(x_test),base3))
stacked_predictions = meta_model.predict(meta_features_test)

In [0]:
stacked_predictions_prob = meta_model.predict_proba(meta_features_test)
stacked_predictions_prob

In [0]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Assuming you have your logistic regression model (log_pol2_Grid) trained and the test data (x_test, y_test)
# Get predicted probabilities for the positive class (class 1)
predicted_probabilities = meta_model.predict_proba(meta_features_test)[:, 1]

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test, predicted_probabilities)

# Calculate AUC (Area Under the Curve)
auc = roc_auc_score(y_test, predicted_probabilities)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()


In [0]:
# Get predicted probabilities for the training data
predicted_probabilities = meta_model.predict_proba(meta_features_test)

# Choose a custom threshold (e.g., 0.6)
custom_threshold = 0.3

# Apply the custom threshold to get binary predictions
binary_predictions = (predicted_probabilities[:, 1] >= custom_threshold).astype(int)

# Calculate F1 score, recall, and precision using binary predictions
f1 = f1_score(y_test, binary_predictions)
recall = recall_score(y_test, binary_predictions)
precision = precision_score(y_test, binary_predictions)
print(f1,recall,precision)


In [0]:

sns.heatmap(confusion_matrix(binary_predictions,y_test),annot=True,fmt='d')

In [0]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

# Create two continuous numerical features
feature1 = np.array([1, 0,0,0,0])
feature2 = np.array([0,1,0,0,0])

# Combine the features into a 2D array
X = np.column_stack((feature1, feature2))

# Create a PolynomialFeatures transformer for quadratic features (degree=2)
poly = PolynomialFeatures(degree=2)

# Apply polynomial features transformation
X_poly = poly.fit_transform(X)

# Display the original features and the polynomial features
print("Original Features:")
print(X)
print("\nPolynomial Features (Quadratic):")
print(X_poly)
